## Summarization (Encoder-Decoder)

In Notebook 1, I worked with GPT-2: a **decoder-only** model. It only ever
looks backward (causal masking) and generates one token at a time, with no
separate "understanding" step before generation.

This notebook uses an **encoder-decoder** model. The key difference:
- The **encoder** reads the entire input text at once, with no masking,
  its only job is to build full understanding of the input.
- The **decoder** generates the summary one token at a time, causally
  masked, just like GPT-2 did.
- At each decoder step, **cross-attention** lets it pull relevant
  information from the encoder's understanding of the original text —
  same Query-Key-Value math as before, just across two different
  sequences instead of one sequence attending to itself.


In [1]:
from transformers import BartTokenizer, BartForConditionalGeneration

# BART is an encoder-decoder model
# "facebook/bart-large-cnn" is specifically fine-tuned for summarization
# (trained on CNN/DailyMail news articles + their summaries)
tokenizer = BartTokenizer.from_pretrained("facebook/bart-large-cnn")
model = BartForConditionalGeneration.from_pretrained("facebook/bart-large-cnn")

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

In [2]:
# Using my own paper abstract as the input text to summarize
text = """
With the development of the industries from the first industrial revolution to what it is in
the current form has created the best platform for the development of healthcare sector from
Healthcare 1.0 when this sector was still a work in progress where the focus was on cost cutting
techniques and to reach a stage where automation could be achieved to some extent and the use
of resources like paper could be decreased as much as possible. However, the biggest hurdle in
achieving this was still the lack of available resources and the stand-alone systems with limited
usage. This challenge was one of the first challenges that was overcome when the industry
oved from Healthcare 1.0 to Healthcare 2.0. Beginning from the year 2000, the healthcare
sector began the usage technologies like clinical imaging for generating the health reports of
patients. There also was a surge in the usage of networking of the IT systems used in healthcare
to provide easier access to anyone in need at any point. While Healthcare 2.0 was crucial in
increasing networking, but 6 years down the line, Healthcare 3.0 brought with itself a life
saving inventions like wearable and implantable devices which have since been crucial in
saving millions of lives across the world. Another important invention of this period was that
of the Electronic Health Records which worked as the perfect solution to the problem of storage
of genomic sequence of people online.
"""

In [3]:
# Tokenize the input text.
# truncation=True just means: if the text is too long for the model's max input size, cut it off rather than error out
import textwrap
inputs = tokenizer(text, return_tensors="pt", truncation=True)

# Generate the summary.
# max_length / min_length control roughly how long the summary should be
summary_ids = model.generate(inputs["input_ids"], max_length=60, min_length=10)

# Decode the generated token IDs back into readable text
summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
print("ORIGINAL LENGTH:", len(text))
print("SUMMARY LENGTH:", len(summary))
print("\n--- ORIGINAL ---")
print(text)
print("\n--- SUMMARY ---")

print(textwrap.fill(summary, width=90))

ORIGINAL LENGTH: 1445
SUMMARY LENGTH: 320

--- ORIGINAL ---

With the development of the industries from the first industrial revolution to what it is in
the current form has created the best platform for the development of healthcare sector from
Healthcare 1.0 when this sector was still a work in progress where the focus was on cost cutting
techniques and to reach a stage where automation could be achieved to some extent and the use
of resources like paper could be decreased as much as possible. However, the biggest hurdle in
achieving this was still the lack of available resources and the stand-alone systems with limited
usage. This challenge was one of the first challenges that was overcome when the industry
oved from Healthcare 1.0 to Healthcare 2.0. Beginning from the year 2000, the healthcare
sector began the usage technologies like clinical imaging for generating the health reports of
patients. There also was a surge in the usage of networking of the IT systems used in healthcar

**Note:** This summary looks almost identical to the start of the original
text, just cut off early: not an actual condensed rewrite. Trying a
different generation strategy (beam search) next to see if that fixes it.

In [4]:
# num_beams=4: beam search explores several possible summaries at once,instead of just greedily picking the single most likely next word each time
# length_penalty=2.0: encourages shorter, more condensed output
# early_stopping=True: stop once a strong summary is found, don't force max_length
summary_ids = model.generate(
    inputs["input_ids"],
    max_length=130,
    min_length=30,
    num_beams=4,
    length_penalty=2.0,
    early_stopping=True
)

summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
print(textwrap.fill(summary, width=90))

The development of the industries from the first industrial revolution to what it is in
its current form has created the best platform for the development of healthcare sector.
Healthcare 1.0 was still a work in progress where the focus was on cost cutting          
    techniques and to reach a stage where automation could be achieved to some extent. The
biggest hurdle in achieving this was still the lack of available resources and the stand-
alone systems with limited usage.


**Note:** This summary extends further into the original text than the
first attempt (now reaches "lack of available resources"), confirming
`max_length=60` was cutting it off too early. But it's still lifting sentences directly from the original rather
than condensing/rewording them.
Testing with news text (as `bart-large-cnn` was actually trained on this) to check if this is a domain-mismatch issue or something else.

In [5]:
news_text = """
Apple unveiled its latest iPhone lineup on Tuesday, featuring a redesigned
camera system and a faster processor built on a new chip architecture.
The company said the new models would be available in stores starting
next week, with pre-orders opening this Friday. Analysts have noted that
this release comes amid slowing smartphone sales globally, as consumers
hold onto their devices for longer periods before upgrading. Apple's
stock rose slightly following the announcement, as investors reacted
positively to the company's emphasis on artificial intelligence features
built into the new operating system update accompanying the devices.
"""

In [6]:
inputs_news = tokenizer(news_text, return_tensors="pt", truncation=True)

summary_ids_news = model.generate(
    inputs_news["input_ids"],
    max_length=130,
    min_length=30,
    num_beams=4,
    length_penalty=2.0,
    early_stopping=True
)

summary_news = tokenizer.decode(summary_ids_news[0], skip_special_tokens=True)
print(textwrap.fill(summary_news, width=90))

Apple unveiled its latest iPhone lineup on Tuesday, featuring a redesignedcamera system
and a faster processor built on a new chip architecture. The company said the new models
would be available in stores starting                next week, with pre-orders opening
this Friday.


##Findings
Tested the same model/settings on both academic text and
news text (which the model was actually trained on). Both produced
near-verbatim copying rather than true abstractive summarization, just
truncated by max_length. This suggests the extractive-leaning behavior
isn't about text domain: it may be more about this specific model's
default generation behavior, or settings I haven't tuned yet.

## Trying a model trained specifically for abstractive summarization



Switching to **facebook/bart-large-xsum** — same BART architecture, but
fine-tuned on the XSum dataset, which specifically uses single,
heavily-reworded one-sentence summaries.

XSum-style summaries are intentionally very short (one sentence)
by design — this isn't meant to cover the whole abstract's detail, just
to test whether the copying problem actually goes away.

In [7]:
from transformers import BartTokenizer, BartForConditionalGeneration

tokenizer_xsum = BartTokenizer.from_pretrained("facebook/bart-large-xsum")
model_xsum = BartForConditionalGeneration.from_pretrained("facebook/bart-large-xsum")

# Tokenize the same healthcare abstract, for a fair comparison
inputs_xsum = tokenizer_xsum(text, return_tensors="pt", truncation=True)

# Same generation settings as our earlier attempts
summary_ids_xsum = model_xsum.generate(
    inputs_xsum["input_ids"],
    max_length=60,
    min_length=15,
    num_beams=4,
    length_penalty=2.0,
    early_stopping=True
)

summary_xsum = tokenizer_xsum.decode(summary_ids_xsum[0], skip_special_tokens=True)
print(textwrap.fill(summary_xsum, width=90))

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.51k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/513 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/309 [00:00<?, ?B/s]

In our series of letters from African journalists, film-maker and columnist Farai Sevenzo
looks at how the healthcare sector has evolved over the years.


## Findings
So, neither model got it right, for opposite reasons. `bart-large-cnn`
was too conservative (just copied), `bart-large-xsum` was too aggressive
and hallucinated a false context (fully reworded, but factually
disconnected from the source).

This traces back to what each model was trained on: `bart-large-cnn`
learned from CNN/DailyMail summaries that stay close to source text, so
it defaults to trimming/copying. `bart-large-xsum` learned from BBC's
heavily-reworded one-line summaries, so it defaults to aggressive
rewording — including BBC's own stylistic patterns ("In our series of
letters from African journalists...") — regardless of whether that
framing actually fits the input. Neither model is "broken"; each is
faithfully reproducing the style of its training data, which doesn't
generalize well to an unrelated domain like an academic abstract.

### What I learned :)

- "Summarization" isn't one fixed behavior — it's whatever style the
  model's training data taught it, which can mean near-copying or full
  rewording depending on the dataset.
- Ran a controlled test (same code, different text domain) to actually
  isolate whether the problem was the input or the model — that's a
  real way to debug, not just guessing.
- Neither model produced a correct summary — one stayed too close to
  the original, the other invented an unrelated framing.
